In [ ]:
import os
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import openpmd_api as io

### Open the series

In [ ]:
BP5_PATH = "/global/cfs/cdirs/m3239/aforment/IFE_AmSC/RHINO/scripts/tmp/rhino_particles.bp5"

series = io.Series(
    BP5_PATH,
    io.Access.read_only, 
    '{"verify_homogeneous_extents": false}'
)

### Auxiliary functions

In [ ]:
def _get_first_snapshot(series):
    """Return the first snapshot from an openPMD RHINO series.

    Parameters
    ----------
    series : openPMD.Series
        OpenPMD RHINO series object.

    Returns
    -------
    object
        First snapshot in the series.

    Raises
    ------
    ValueError
        If the series does not contain any snapshots.
    """
    snapshots = series.snapshots()
    if not snapshots:
        raise ValueError("The series does not contain any snapshots.")
    return snapshots[0]

def _validate_species(it, species):
    """Validate that a species exists in a snapshot and is not ``'Times'``.

    Parameters
    ----------
    it : object
        Snapshot object from an openPMD RHINO series.
    species : str
        Species name to validate.

    Returns
    -------
    list[str]
        List of available species names.

    Raises
    ------
    ValueError
        If the species is invalid.
    """
    avail_species = [sp for sp in it.particles if sp != "Times"]
    if species == "Times" or species not in avail_species:
        raise ValueError(
            f"Invalid species {species!r}. Available species are: {avail_species}"
        )
    return avail_species

def _validate_subsystem(it, species, subsystem):
    """Validate that a subsystem exists for a given species.

    Parameters
    ----------
    it : object
        Snapshot object from an openPMD RHINO series.
    species : str
        Species name whose subsystems are being queried.
    subsystem : str
        Subsystem name to validate.

    Returns
    -------
    list[str]
        List of available subsystem names for the species.

    Raises
    ------
    ValueError
        If the subsystem is invalid.
    """
    avail_subsystems = [sub for sub in it.particles[species]["subsystems"]]
    if subsystem not in avail_subsystems:
        raise ValueError(
            f"Invalid subsystem {subsystem!r}. "
            f"Available subsystems for species {species!r} are: {avail_subsystems}"
        )
    return avail_subsystems

def print_summary(series):
    """Print a human-readable summary of an openPMD RHINO series.

    The summary includes:
    - series-level metadata
    - metadata for the first snapshot
    - available particle records and record components in that snapshot

    Parameters
    ----------
    series : openPMD.Series
        OpenPMD RHINO series object to inspect.

    Returns
    -------
    None
        This function prints information to standard output.

    Raises
    ------
    ValueError
        If the series does not contain any snapshots.
    """
    it = _get_first_snapshot(series)

    print("===============")
    print("Series metadata")
    print("===============")

    for a in series.attributes:
        print(f"{a}={series.get_attribute(a)}")

    print("\t==================")
    print("\tIteration metadata")
    print("\t==================")
    for a in it.attributes:
        print(f"\t{a}={it.get_attribute(a):<20}")

    print("\t\t==============")
    print("\t\tAvailable data")
    print("\t\t==============")
    for p in it.particles:
        print(f"\t\t{p}")
        for r in it.particles[p]:
            print(f"\t\t\t{r}")
            for c in it.particles[p][r]:
                print(f"\t\t\t\t{c}")

def get_times(series):
    """Return the time array stored in an openPMD RHINO series.

    This function reads the scalar ``Times/data`` record from the first snapshot
    in the series and returns it as a NumPy array. Time values are expected to
    be in days.

    Parameters
    ----------
    series : openPMD.Series
        OpenPMD RHINO series object containing time data.

    Returns
    -------
    numpy.ndarray
        One-dimensional array of time values in days.

    Raises
    ------
    ValueError
        If the series does not contain any snapshots.
    """
    it = _get_first_snapshot(series)
    times = it.particles["Times"]["data"][io.Record_Component.SCALAR].load_chunk()
    return times

def get_steady_state(series, species="Tritium", subsystem=None):
    """Return the steady-state mass inventory for a species.

    This function loads the ``mass_steady`` data for the requested species from
    the first snapshot in the series. If a subsystem name is provided, only the
    value for that subsystem is returned.

    Parameters
    ----------
    series : openPMD.Series
        OpenPMD RHINO series object containing steady-state mass data.
    species : str, optional
        Name of the species to query. Default is ``"Tritium"``.
    subsystem : str or None, optional
        Name of a subsystem for which to extract the steady-state value.
        If ``None``, the full array is returned.

    Returns
    -------
    numpy.ndarray or scalar
        If ``subsystem is None``, returns the full steady-state mass array.
        Otherwise returns the steady-state mass value for the selected subsystem.

    Raises
    ------
    ValueError
        If the series does not contain any snapshots, if the species is invalid,
        or if the subsystem is not found.
    """
    it = _get_first_snapshot(series)
    _validate_species(it, species)

    mass_steady = it.particles[species]["mass_steady"][io.Record_Component.SCALAR]
    mass_steady_data = mass_steady.load_chunk()
    series.flush()

    if subsystem is None:
        return mass_steady_data

    _validate_subsystem(it, species, subsystem)
    subsystem_id = it.particles[species]["subsystems"][subsystem].get_attribute("id")
    return mass_steady_data[subsystem_id]


def get_time_inventory(series, species="Tritium", subsystem=None):
    """Return the time-dependent mass inventory for a species.

    This function loads the ``mass`` data for the requested species from the
    first snapshot in the series. If a subsystem name is provided, only the
    time history for that subsystem is returned.

    Parameters
    ----------
    series : openPMD.Series
        OpenPMD RHINO series object containing mass inventory data.
    species : str, optional
        Name of the species to query. Default is ``"Tritium"``.
    subsystem : str or None, optional
        Name of a subsystem for which to extract the time-dependent mass.
        If ``None``, the full array is returned.

    Returns
    -------
    numpy.ndarray
        If ``subsystem is None``, returns the full mass array, typically indexed
        by subsystem and time. Otherwise returns the one-dimensional time series
        for the selected subsystem.

    Raises
    ------
    ValueError
        If the series does not contain any snapshots, if the species is invalid,
        or if the subsystem is not found.
    """
    it = _get_first_snapshot(series)
    _validate_species(it, species)

    mass = it.particles[species]["mass"][io.Record_Component.SCALAR]
    mass_data = mass.load_chunk()
    series.flush()

    if subsystem is None:
        return mass_data

    _validate_subsystem(it, species, subsystem)
    subsystem_id = it.particles[species]["subsystems"][subsystem].get_attribute("id")
    return mass_data[subsystem_id, :]

def get_subsystem_id(series, subsystem, species="Tritium"):
    """Return the integer ID associated with a subsystem name.

    Parameters
    ----------
    series : openPMD.Series
        OpenPMD RHINO series object containing subsystem metadata.
    subsystem : str
        Name of the subsystem to look up.
    species : str, optional
        Name of the species whose subsystem mapping should be used.
        Default is ``"Tritium"``.

    Returns
    -------
    int
        Integer ID associated with the requested subsystem.

    Raises
    ------
    ValueError
        If the series does not contain any snapshots, if the species is invalid,
        or if the subsystem is not available for that species.
    """
    it = _get_first_snapshot(series)
    _validate_species(it, species)
    _validate_subsystem(it, species, subsystem)
    return it.particles[species]["subsystems"][subsystem].get_attribute("id")


def get_subsystem_name(series, subsystem_id, species="Tritium"):
    """Return the subsystem name associated with a subsystem ID.

    Parameters
    ----------
    series : openPMD.Series
        OpenPMD RHINO series object containing subsystem metadata.
    subsystem_id : int
        Integer subsystem ID to look up.
    species : str, optional
        Name of the species whose subsystem mapping should be used.
        Default is ``"Tritium"``.

    Returns
    -------
    str
        Name of the subsystem associated with the given ID.

    Raises
    ------
    ValueError
        If the series does not contain any snapshots, if the species is invalid,
        if the subsystem ID is out of range, or if no subsystem matches the ID.
    """
    it = _get_first_snapshot(series)
    _validate_species(it, species)

    n_subsystems = len(it.particles[species]["subsystems"])
    if subsystem_id not in range(n_subsystems):
        raise ValueError(
            f"Invalid subsystem_id {subsystem_id!r}. "
            f"Valid IDs are: {list(range(n_subsystems))}"
        )

    for k, v in it.particles[species]["subsystems"].items():
        if v.get_attribute("id") == subsystem_id:
            return k

    raise ValueError(
        f"No subsystem found with id {subsystem_id!r} for species {species!r}."
    )

def get_subsystems_dict(series, species="Tritium"):
    """Return a dictionary mapping subsystem names to subsystem IDs.

    Parameters
    ----------
    series : openPMD.Series
        OpenPMD RHINO series object containing subsystem metadata.
    species : str, optional
        Name of the species whose subsystem mapping should be returned.
        Default is ``"Tritium"``.

    Returns
    -------
    dict[str, int]
        Dictionary whose keys are subsystem names and whose values are
        subsystem IDs.

    Raises
    ------
    ValueError
        If the series does not contain any snapshots or if the requested
        species is not available.
    """
    it = _get_first_snapshot(series)
    _validate_species(it, species)

    subsystems = {}
    for k, v in it.particles[species]["subsystems"].items():
        subsystems[k] = v.get_attribute("id")

    return subsystems
    

In [ ]:
times = get_times(series)
subsystems_dict = get_subsystems_dict(series, species="Tritium")
Tss = get_steady_state(series, species="Tritium")
Tsubs = list(subsystems_dict.keys())

In [ ]:
fig,ax = plt.subplots(ncols=1, nrows=1,figsize=(6,6))
colors = sns.color_palette("hls", len(Tsubs))
ax.barh(Tsubs, Tss, label="T", color=colors)
ax.set_xlabel('mass [g]')
ax.set_title('T steady state')
ax.set_xscale('log')
plt.show()

In [ ]:
T_ts = get_time_inventory(series, species="Tritium")
row, col = np.unravel_index(np.argmax(T_ts), T_ts.shape)
print(get_subsystem_name(series, row))

In [ ]:
T_gasdetrit = get_time_inventory(series, species="Tritium", subsystem='Gas_Detrit')
Tss_gasdetrit = get_steady_state(series, species="Tritium", subsystem='Gas_Detrit')
T_genbox = get_time_inventory(series, species="Tritium", subsystem='Gen_Box')
Tss_genbox = get_steady_state(series, species="Tritium", subsystem='Gen_Box')

fig,ax = plt.subplots(ncols=1, nrows=1,figsize=(10,6))

ax.plot(times, T_gasdetrit, label="Gas Detrit")
ax.scatter(times[-1], Tss_gasdetrit)
ax.plot(times, T_genbox, label="Gen Box")
ax.scatter(times[-1], Tss_genbox)

print(Tss_genbox)
ax.set_ylabel('time [d]')
ax.set_ylabel('mass [g]')
ax.set_title('T time series')
ax.set_yscale('log')
ax.set_xscale('log')
ax.legend()
plt.show()

In [ ]:
fig,ax = plt.subplots(ncols=1, nrows=1,figsize=(10,6))
i=0
for k, v in subsystems_dict.items():
    
    ax.plot(times[::100], T_ts[int(v),::100] , label=k, color=colors[i])
    ax.scatter(times[-1], Tss[int(v)], color=colors[i])
    i += 1
    
ax.set_ylabel('time [d]')
ax.set_ylabel('mass [g]')
ax.set_title('T time series')
ax.set_yscale('log')
ax.set_xscale('log')
ax.legend()
plt.show()